# 05 - The pipeline tool - ChurnLens

Everything so far produced *artifacts*. This notebook turns them into a **product**:

> hand it a file -> it says which of the five lenses that data supports (and why not,
> when not) -> scores every supported lens -> returns one enriched table.

Design guarantees (all demonstrated below): named human-readable failures, no silent
column drops, unseen categories tolerated, same fitted transformers at scoring time
(never refit), deterministic output.

In [1]:
import sys
from pathlib import Path

# Make src/ importable whether Jupyter starts in the project root or in Code/
for candidate in (Path.cwd(), Path.cwd() / "Code", Path.cwd().parent):
    if (candidate / "src").exists():
        sys.path.insert(0, str(candidate))
        break

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)

from src import pipeline
from src.data_loader import load_telco, load_retail

path = pipeline.write_contracts()
print(f"schema contracts -> {path}")

schema contracts -> /Users/prajwalnerkar/Prajwal/Projects/CustomerChurnAnalysis/models/contracts/contracts.json


## 1. Happy path - a Telco-shaped file

Note what goes in: the **raw** frame (string `TotalCharges` and all) - coercion is the
tool's job, recorded in the report, not the caller's.

In [2]:
telco_raw = load_telco(clean=False).drop(columns=["Churn"]).sample(8, random_state=42)
report = pipeline.validate(telco_raw)
for lens, info in report["lenses"].items():
    print(f"Lens {lens}: {info['message']}")
print("ignored columns:", report["ignored_columns"])

Lens A: ready (Churn prediction)
Lens B: ready (Survival (expected remaining tenure))
Lens C: disabled - missing: treatment_group, conversion
Lens D: ready (Segmentation)
Lens E: disabled - missing: CustomerID, InvoiceDate
ignored columns: []


In [3]:
scored, report = pipeline.score(telco_raw)
print("applied:", report["applied"], "| errors:", report.get("errors") or "none")
scored[["customerID", "churn_probability", "risk_band", "expected_remaining_tenure",
        "segment", "clv_simple", "revenue_at_risk", "recommended_action"]]

applied: ['A', 'B', 'D', 'E-simplified'] | errors: none


,customerID,churn_probability,risk_band,expected_remaining_tenure,segment,clv_simple,revenue_at_risk,recommended_action
185,1024-GUALD,0.8383,Critical,27.8,1,689.44,577.96,Offer 1-year contract discount
2715,0484-JPBRU,0.2324,Low,10.1,1,255.02,59.27,Monitor
3825,3620-EHIMZ,0.0570,Low,13.8,1,267.03,15.22,Monitor
1807,6910-HADCM,0.8446,Critical,20.8,1,1588.08,1341.29,Offer 1-year contract discount
132,8587-XYZSF,0.0779,Low,0.0,0,0.00,0.00,Monitor
1263,6818-WOBHJ,0.4820,Medium,0.0,0,0.00,0.00,Offer 1-year contract discount
3732,3082-YVEKW,0.0764,Low,46.0,0,3548.90,271.14,Monitor
1672,4737-AQCPU,0.0591,Low,0.0,0,0.00,0.00,Monitor


## 2. Broken data - the failure must be *named*

Drop `tenure`, misspell `gender`. Lenses A/B/D must disable themselves and say
exactly which columns are missing - no stack trace, no silent guess.

In [4]:
broken = telco_raw.drop(columns=["tenure"]).rename(columns={"gender": "Gender"})
report = pipeline.validate(broken)
for lens, info in report["lenses"].items():
    print(f"Lens {lens}: {info['message']}")

Lens A: disabled - missing: gender, tenure
Lens B: disabled - missing: gender, tenure
Lens C: disabled - missing: treatment_group, conversion
Lens D: disabled - missing: gender, tenure
Lens E: disabled - missing: CustomerID, InvoiceDate


## 3. Foreign-but-mappable data + unseen categories

Two robustness checks: (a) a file whose *values* were never seen in training
(new payment method label) - the one-hot encoder was built with
`handle_unknown="ignore"`, so scoring proceeds; (b) a transaction log routed
automatically to lens E by `score_any`.

In [5]:
weird = telco_raw.copy()
weird.loc[:, "PaymentMethod"] = "Crypto wallet"     # category unseen in training
scored_w, rep_w = pipeline.score(weird)
print("unseen category handled, applied:", rep_w["applied"])
scored_w[["churn_probability", "risk_band"]].head(3)

unseen category handled, applied: ['A', 'B', 'D', 'E-simplified']


/opt/anaconda3/envs/customerchurn/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/opt/anaconda3/envs/customerchurn/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


,churn_probability,risk_band
185,0.7530,High
2715,0.2335,Low
3825,0.0570,Low


In [6]:
tx = load_retail().sample(50_000, random_state=42)
scored_tx, rep_tx = pipeline.score_any(tx)
print("routed to:", rep_tx["applied"])
scored_tx.head()

routed to: ['E']


,CustomerID,Recency,Frequency,Monetary,R,F,M,segment,frequency,recency,T,monetary_value,clv
0,12346,647,3,29.45,1,3,2,At risk,1.0,74.0,721.0,5.950,5.998397
1,12347,2,6,200.03,5,4,4,Champions,5.0,402.0,404.0,37.906,158.955373
2,12348,358,2,83.76,2,2,3,Hibernating,1.0,80.0,438.0,63.600,56.150610
3,12349,19,3,476.73,5,3,5,Regular,2.0,571.0,589.0,221.265,298.459126
4,12350,310,1,47.40,2,1,2,Hibernating,0.0,0.0,310.0,0.000,0.000000


## 4. The CLI - same engine, no notebook required

This is goal #1 from the project brief: *"tomorrow someone can churn-analyze new data
without me"*. They run one command.

In [7]:
sample_path = Path("../Dataset/processed/cli_demo.csv")
load_telco(clean=False).drop(columns=["Churn"]).sample(100, random_state=1).to_csv(
    sample_path, index=False)
import subprocess
out = subprocess.run(
    [sys.executable, "-m", "src.pipeline", "score",
     "--input", str(sample_path)],
    capture_output=True, text=True, cwd=Path.cwd())
print(out.stdout or out.stderr)

  Lens A: ready (Churn prediction)
  Lens B: ready (Survival (expected remaining tenure))
  Lens C: disabled - missing: treatment_group, conversion
  Lens D: ready (Segmentation)
  Lens E: disabled - missing: CustomerID, InvoiceDate
  Applied: ['A', 'B', 'D', 'E-simplified']
  Wrote 100 rows -> ../Dataset/processed/cli_demo_scored.csv



## 5. Retraining path - the tool improves instead of aging

`python -m src.pipeline train --data new_labeled.csv` refits Lens A, timestamps the
artifact, and updates the default pointer. (Not executed here - it is a several-minute
run; the CLI help shows the interface.)

In [8]:
out = subprocess.run([sys.executable, "-m", "src.pipeline", "--help"],
                     capture_output=True, text=True, cwd=Path.cwd())
print(out.stdout)

usage: churn-pipeline [-h] {score,train,contracts} ...

ChurnLens: validate, score or retrain on a customer data file.

positional arguments:
  {score,train,contracts}
    score               validate + score a file with every available lens
    train               retrain Lens A on a labeled snapshot file
    contracts           write schema contracts to models/contracts/

options:
  -h, --help            show this help message and exit



## 6. The app - ChurnLens UI

```bash
# from the project root
.venv/bin/streamlit run app/streamlit_app.py
```

- **Score tab** - single-customer what-if form ("customer belongs to demographic X,
  has features Y" -> probability gauge, remaining tenure, CLV, revenue at risk,
  recommended action) or batch upload with the lens chips + validation report.
- **Targeting tab** - uplift, when campaign data is present.
- **Model info tab** - the model cards from `models/metrics.json`.

**Deployment note (Streamlit Community Cloud):** `models/` is gitignored, so a cloud
deploy must either (a) rebuild artifacts on first boot from a bundled sample, or
(b) pull them from private storage. Decision deferred to the deploy step - locally,
everything above just works.

---
### The demo script (for interviews)
1. Upload a fresh Telco-style CSV -> chips A/B/D/E-simplified light up -> sorted
   Revenue-at-Risk list.
2. Upload the same file with `tenure` deleted -> watch the tool *explain itself*
   instead of erroring.
3. Upload a transaction log -> the tool reroutes to CLV/RFM automatically.
4. One CLI command, same results - "the notebook is the lab; this is the product."
